# WCC Data Loading
Loads all data for the Wominjeka Coffee Co vehicle routing project.
- **Real data**: 34 Melbourne cafe locations, distance matrix (km), time matrix (min)
- **Synthetic data**: product catalogue, per-cafe demands, van fleet, perishability params
- **Depot**: Peter Hall Building, UniMelb

Run this block at the top of the main project notebook.

In [1]:
import pandas as pd
import numpy as np

SYNTHETIC_DATA_DIR = '../data/synthetic'  # adjust path as needed
REAL_DATA_DIR = '../data/real'

## Load Products

In [2]:
products = pd.read_csv(f'{SYNTHETIC_DATA_DIR}/products.csv')
P = products['product_id'].tolist()  # set of products
P_milk = products[products['perishable'] == True]['product_id'].tolist()
P_beans = products[products['perishable'] == False]['product_id'].tolist()

# Parameter dictionaries
revenue = dict(zip(products['product_id'], products['revenue_per_unit']))  # r_p
cost = dict(zip(products['product_id'], products['cost_per_unit']))
margin = dict(zip(products['product_id'], products['margin_per_unit']))
weight = dict(zip(products['product_id'], products['weight_per_unit_kg']))  # w_p

print(f'Products: {len(P)} ({len(P_milk)} milk, {len(P_beans)} beans)')
products

Products: 7 (4 milk, 3 beans)


,product_id,product_name,category,unit,weight_per_unit_kg,revenue_per_unit,cost_per_unit,perishable,shelf_life_hours,margin_per_unit
0,milk_full,Full Cream Milk,milk,litres,1.03,2.8,1.4,True,8,1.4
1,milk_oat,Oat Milk,milk,litres,1.03,4.2,2.1,True,8,2.1
2,milk_soy,Soy Milk,milk,litres,1.02,3.5,1.6,True,10,1.9
3,milk_almond,Almond Milk,milk,litres,1.02,4.5,2.2,True,10,2.3
4,bean_house,House Blend Beans,beans,kg,1.00,18.0,9.5,False,999,8.5
5,bean_single,Single Origin Beans,beans,kg,1.00,28.0,16.0,False,999,12.0
6,bean_decaf,Decaf Beans,beans,kg,1.00,22.0,12.5,False,999,9.5


## Load Cafes and Demands

In [3]:
cafes = pd.read_csv(f'{SYNTHETIC_DATA_DIR}/cafes.csv')
demands_df = pd.read_csv(f'{SYNTHETIC_DATA_DIR}/demands.csv')

# Node mapping: 0 = depot, 1..n = cafes
cafe_ids = cafes['cafe_id'].tolist()
n = len(cafe_ids)
V = list(range(n + 1))  # all nodes
V_cafes = list(range(1, n + 1))  # cafe nodes only

cafe_to_node = {cid: i+1 for i, cid in enumerate(cafe_ids)}
node_to_cafe = {i+1: cid for i, cid in enumerate(cafe_ids)}
node_to_name = {0: 'Depot (Peter Hall)'}
node_to_name.update({i+1: cafes.loc[i, 'cafe_name'] for i in range(n)})

# Demand dictionary: d[i][p] = demand of node i for product p
d = {}
for _, row in demands_df.iterrows():
    node = cafe_to_node[row['cafe_id']]
    if node not in d:
        d[node] = {}
    d[node][row['product_id']] = row['daily_demand']

print(f'Cafes: {n}')
print(f'Nodes: {len(V)} (depot + {n} cafes)')
print(f'\nFirst 5 cafes:')
cafes.head()

Cafes: 34
Nodes: 35 (depot + 34 cafes)

First 5 cafes:


,cafe_id,cafe_name,latitude,longitude,size
0,cafe_01,Proud Mary Coffee,-37.802338,144.985172,small
1,cafe_02,Patricia Coffee Brewers,-37.814693,144.958238,large
2,cafe_03,Seven Seeds Coffee Roasters,-37.802971,144.959003,medium
3,cafe_04,Industry Beans Fitzroy,-37.794811,144.978134,medium
4,cafe_05,Market Lane Coffee HQ,-37.776348,144.972123,small


## Load Distance and Time Matrices (Real Data)

In [4]:
dist_df = pd.read_csv(f'{REAL_DATA_DIR}/distance_matrix.csv', index_col=0)
time_df = pd.read_csv(f'{REAL_DATA_DIR}/time_matrix.csv', index_col=0)

# Node labels in matrix: 'depot', 'cafe_01', 'cafe_02', ...
node_labels = ['depot'] + cafe_ids

# Travel cost: distance_km * fuel_cost_per_km (use average across fleet)
avg_fuel = 0.45  # $/km average

# c[i,j] = travel cost from node i to node j
c = {}
for i in V:
    for j in V:
        if i != j:
            c[i, j] = round(dist_df.iloc[i, j] * avg_fuel, 4)

# dist_km[i,j] = distance in km
dist_km = {}
for i in V:
    for j in V:
        if i != j:
            dist_km[i, j] = round(dist_df.iloc[i, j], 4)

# t_travel[i,j] = travel time in hours (convert from minutes)
t_travel = {}
for i in V:
    for j in V:
        if i != j:
            t_travel[i, j] = round(time_df.iloc[i, j] / 60, 4)  # min -> hours

# Arcs
A = [(i, j) for i in V for j in V if i != j]

print(f'Arcs: {len(A)}')
print(f'\nSample (depot to Proud Mary Coffee):')
print(f'  Distance: {dist_km[0,1]:.1f} km')
print(f'  Travel time: {t_travel[0,1]*60:.1f} min')
print(f'  Travel cost: ${c[0,1]:.2f}')

Arcs: 1190

Sample (depot to Proud Mary Coffee):
  Distance: 2.4 km
  Travel time: 5.5 min
  Travel cost: $1.08


## Load Van Fleet

In [5]:
vans_df = pd.read_csv(f'{SYNTHETIC_DATA_DIR}/vans.csv')

num_vans = 3  # default — change for experiments
K = list(range(num_vans))
Q = {k: vans_df.loc[k, 'capacity_kg'] for k in K}  # Q_k
fuel_cost_per_van = {k: vans_df.loc[k, 'fuel_cost_per_km'] for k in K}

print(f'Using {num_vans} vans')
print(f'Capacities: {Q}')
vans_df.head(num_vans)

Using 3 vans
Capacities: {0: 600, 1: 600, 2: 700}


,van_id,capacity_kg,fuel_cost_per_km,has_refrigeration
0,van_1,600,0.45,True
1,van_2,600,0.45,True
2,van_3,700,0.50,True


## Load Perishability Parameters

In [6]:
perish_df = pd.read_csv(f'{SYNTHETIC_DATA_DIR}/perishability_params.csv')
perish_params = dict(zip(perish_df['parameter'], perish_df['value']))

T_max = perish_params['T_max_hours']  # hours
service_time_hr = perish_params['service_time_minutes'] / 60  # convert to hours
loading_time_hr = perish_params['loading_time_minutes'] / 60

print(f'T_max: {T_max} hours')
print(f'Service time per cafe: {service_time_hr*60:.0f} min ({service_time_hr:.4f} hr)')
print(f'Loading time at depot: {loading_time_hr*60:.0f} min ({loading_time_hr:.4f} hr)')
perish_df

T_max: 3.0 hours
Service time per cafe: 5 min (0.0833 hr)
Loading time at depot: 15 min (0.2500 hr)


,parameter,value,description
0,T_max_hours,3.00,Default max route duration for vans carrying m...
1,T_max_hours_alt1,2.00,Tight scenario for sensitivity analysis
2,T_max_hours_alt2,4.00,Relaxed scenario for sensitivity analysis
3,milk_spoilage_cost_per_hour,0.15,Cost penalty per litre of milk per hour on the...
4,loading_time_minutes,15.00,Time to load a van at the depot (minutes)
5,service_time_minutes,5.00,Time to unload/service at each cafe (minutes)


## Sanity Check

In [7]:
total_weight = sum(d[i][p] * weight[p] for i in V_cafes for p in P)
total_capacity = sum(Q[k] for k in K)
total_revenue = sum(d[i][p] * revenue[p] for i in V_cafes for p in P)
total_milk_L = sum(d[i][p] for i in V_cafes for p in P_milk)
total_beans_kg = sum(d[i][p] for i in V_cafes for p in P_beans)

print(f'=== Data Summary ===')
print(f'Cafes: {n} | Products: {len(P)} | Vans: {num_vans}')
print(f'\nDemand:')
print(f'  Total milk: {total_milk_L} litres')
print(f'  Total beans: {total_beans_kg} kg')
print(f'  Total weight: {total_weight:.0f} kg')
print(f'  Total revenue: ${total_revenue:.0f}')
print(f'\nCapacity:')
print(f'  Fleet capacity: {total_capacity} kg')
print(f'  Utilisation: {total_weight/total_capacity*100:.0f}%')
print()
if total_weight > total_capacity:
    print('WARNING: Demand exceeds capacity. Not all cafes can be served.')
else:
    print('OK: Capacity sufficient to serve all cafes.')

=== Data Summary ===
Cafes: 34 | Products: 7 | Vans: 3

Demand:
  Total milk: 1055 litres
  Total beans: 123 kg
  Total weight: 1208 kg
  Total revenue: $6093

Capacity:
  Fleet capacity: 1900 kg
  Utilisation: 64%

OK: Capacity sufficient to serve all cafes.
